# Eigene Piper-Stimme trainieren (Kaggle / Google Colab)

Dieses Notebook macht aus der ZIP-Datei des **Stimmen-Studios** (`wavs/` + `metadata.csv`) einen **Stimmenordner für die Chat-App**:
Piper installieren, eine deutsche Ausgangsstimme laden, auf deine Aufnahmen feinabstimmen, nach ONNX exportieren und für die App verpacken.

**Vorher wissen:**
- Du brauchst eine **GPU** (Kaggle: *Settings → Accelerator → GPU*, Colab: *Laufzeit → Laufzeittyp ändern → T4*). Ohne GPU bricht Zelle 2 ab.
- **Deine Aufnahmen werden zum Anbieter hochgeladen** (Google bzw. Kaggle). Das ist der Preis für die gemietete GPU.
- Mindestens **30 Minuten** Aufnahmen, besser 60. Mit weniger wird die Stimme nicht gut.
- Dauer: einige Stunden. Das Notebook bricht nach `MAX_HOURS` ab und speichert vorher; du kannst danach exportieren oder mit `RESUME = True` weitermachen.
- Dieses Notebook folgt der Piper-Anleitung (`OHF-Voice/piper1-gpl`, `docs/TRAINING.md`). **Es konnte nicht auf einer GPU getestet werden.** Wenn eine Zelle mit einem Fehler
  abbricht, kopiere die Fehlermeldung und gib sie weiter; die Versionen der Bibliotheken auf Kaggle/Colab ändern sich.

**Ablauf:** Zellen von oben nach unten ausführen. Nur Zelle 1 enthält Einstellungen.

## 1. Einstellungen

In [ ]:
# --- Einstellungen: nur hier etwas ändern ---------------------------------------------------------------------------------
NAME = "meine-stimme"                 # Name der Stimme (Buchstaben, Ziffern, - und _)
LANG = "de_DE"                        # Sprache mit Land: de_DE, en_US, en_GB, fr_FR, es_ES, it_IT, ...
ESPEAK_VOICE = "de"                   # Aussprache: de, en-us, en-gb, fr, es, it, ...
BASE = "de/de_DE/thorsten/medium"     # Ausgangsstimme (Ordner in huggingface.co/datasets/rhasspy/piper-checkpoints, nur "medium")
EXTRA_EPOCHS = 600                    # Epochen zusätzlich zum Ausgangs-Checkpoint (Epoche = ein Durchlauf durch alle Aufnahmen)
MAX_HOURS = 8                         # Sicherheitsgrenze: danach wird gespeichert und beendet (Kaggle/Colab trennen nach ca. 9-12 h)
INT8 = True                           # fertiges Modell auf int8 umrechnen (kleiner, auf dem Handy schneller)
RESUME = False                        # True = mit dem letzten Zwischenstand dieses Laufs weitermachen
DATASET_ZIP = ""                      # Pfad der ZIP aus dem Stimmen-Studio; leer = automatisch suchen / bei Colab hochladen
TEST_SENTENCE = "Hallo, das ist meine eigene Stimme. Wie klingt sie für dich?"

## 2. GPU prüfen

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "Keine GPU gefunden. Kaggle: Settings → Accelerator → GPU. Colab: Laufzeit → Laufzeittyp ändern → T4-GPU. Danach die Sitzung neu starten."
)
GPU = torch.cuda.get_device_name(0)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
BATCH = 32 if GPU_GB >= 20 else 16
print(f"GPU: {GPU}, {GPU_GB:.0f} GB -> Batchgröße {BATCH}")

## 3. Piper und Werkzeuge installieren (einige Minuten)

In [ ]:
import os, subprocess, sys

def sh(cmd, check=True):
    """Shell-Befehl mit laufender Ausgabe."""
    print("$", cmd)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError(f"Befehl fehlgeschlagen ({p.returncode}): {cmd}")

sh("apt-get -qq update && apt-get -qq install -y espeak-ng build-essential cmake ninja-build")
if not os.path.isdir("/content/piper1-gpl") and not os.path.isdir("/kaggle/working/piper1-gpl"):
    base = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
    os.chdir(base)
    sh("git clone https://github.com/OHF-voice/piper1-gpl.git")
else:
    base = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
REPO = f"{base}/piper1-gpl"
os.chdir(REPO)
sh("python3 -m pip install -q -e '.[train]'")
sh("./build_monotonic_align.sh")
sh("python3 -m pip install -q onnx onnxruntime huggingface_hub sherpa-onnx==1.13.8")
WORK = f"{base}/work"
os.makedirs(WORK, exist_ok=True)
print("Piper bereit:", REPO)

## 4. Hilfsfunktionen

In [ ]:
import glob, json, os, re, shutil, wave, zipfile

def locate_dataset(root):
    """Ordner mit metadata.csv unterhalb von root (die ZIP darf einen Oberordner haben)."""
    for path in [root] + sorted(glob.glob(os.path.join(root, "**"), recursive=True)):
        if os.path.isdir(path) and os.path.isfile(os.path.join(path, "metadata.csv")):
            return path
    raise FileNotFoundError("In der ZIP fehlt metadata.csv (erwartet: metadata.csv und Ordner wavs/).")

def dataset_info(root):
    """(Anzahl, Minuten, Fehlerliste) für <root>/metadata.csv und <root>/wavs/."""
    errors, total, count = [], 0.0, 0
    for line in open(os.path.join(root, "metadata.csv"), encoding="utf-8"):
        line = line.strip()
        if not line:
            continue
        name, _, text = line.partition("|")
        wav = os.path.join(root, "wavs", name if name.lower().endswith(".wav") else name + ".wav")
        if not text.strip():
            errors.append(f"{name}: kein Text")
        if not os.path.isfile(wav):
            errors.append(f"{name}: Datei fehlt")
            continue
        with wave.open(wav) as w:
            if w.getframerate() != 22050 or w.getnchannels() != 1:
                errors.append(f"{name}: {w.getframerate()} Hz, {w.getnchannels()} Kanäle (erwartet 22050 Hz, mono)")
            total += w.getnframes() / w.getframerate()
        count += 1
    return count, total / 60, errors

def newest_checkpoint(*roots):
    files = [f for r in roots for f in glob.glob(os.path.join(r, "**", "*.ckpt"), recursive=True)]
    return max(files, key=os.path.getmtime) if files else None

LANGUAGE_NAMES = {"de": "German", "en": "English", "fr": "French", "es": "Spanish", "it": "Italian", "nl": "Dutch", "pt": "Portuguese",
                  "pl": "Polish", "ru": "Russian", "uk": "Ukrainian", "cs": "Czech", "sv": "Swedish", "da": "Danish", "fi": "Finnish"}

def tokens_from_config(config):
    """tokens.txt für sherpa-onnx (wie das offizielle Umwandlungsskript)."""
    lines = []
    for symbol, ident in config["phoneme_id_map"].items():
        if symbol == "\n":
            continue
        if isinstance(ident, list):
            ident = ident[0]
        lines.append(f"{symbol} {ident}")
    return "\n".join(lines) + "\n"

def model_metadata(config, lang):
    """Metadaten, die sherpa-onnx im Modell erwartet (wie das offizielle add_meta_data.py)."""
    rate = config["audio"]["sample_rate"]
    rate = 22050 if rate == 22500 else rate
    voice = config.get("lang_code") or config["espeak"]["voice"]
    voice = "pt" if voice == "pt-PT" else voice
    code = lang.split("_")[0].lower()
    return {"model_type": "vits", "comment": "piper", "language": LANGUAGE_NAMES.get(code, code), "voice": voice, "version": 1,
            "has_espeak": 1, "has_g2pw": 0, "n_speakers": config.get("num_speakers", 1), "sample_rate": rate}

def package_voice(onnx_path, config_path, name, lang, out_dir, espeak_dir=None, int8=True):
    """Legt <out_dir>/<lang>-<name>-<qualität>/ mit Modell, tokens.txt, .onnx.json (und espeak-ng-data) an. Liefert den Ordner."""
    import onnx
    config = json.load(open(config_path, encoding="utf-8"))
    quality = config["audio"].get("quality") or "medium"
    folder = os.path.join(out_dir, f"{lang}-{name}-{quality}")
    os.makedirs(folder, exist_ok=True)
    target = os.path.join(folder, f"{lang}-{name}-{quality}.onnx")
    if int8:
        from onnxruntime.quantization import QuantType, quantize_dynamic
        quantize_dynamic(model_input=onnx_path, model_output=target, weight_type=QuantType.QUInt8)
    else:
        shutil.copyfile(onnx_path, target)
    model = onnx.load(target)
    while len(model.metadata_props):
        model.metadata_props.pop()
    for key, value in model_metadata(config, lang).items():
        prop = model.metadata_props.add()
        prop.key, prop.value = key, str(value)
    onnx.save(model, target)
    open(os.path.join(folder, "tokens.txt"), "w", encoding="utf-8", newline="\n").write(tokens_from_config(config))
    shutil.copyfile(config_path, target + ".json")
    if espeak_dir:
        shutil.copytree(espeak_dir, os.path.join(folder, "espeak-ng-data"), dirs_exist_ok=True)
    return folder
print("Hilfsfunktionen geladen")

## 5. Aufnahmen bereitstellen

- **Kaggle:** Beim Notebook rechts *Add Input → Upload* die ZIP als neuen Datensatz hochladen (privat lassen), dann diese Zelle ausführen.
- **Colab:** Die Zelle öffnet einen Datei-Dialog. Alternativ die ZIP links im Dateibereich nach `/content` ziehen und `DATASET_ZIP` in Zelle 1 setzen.

In [ ]:
zip_path = DATASET_ZIP
if not zip_path:
    candidates = glob.glob("/kaggle/input/**/*.zip", recursive=True) + glob.glob("/content/*.zip")
    zip_path = candidates[0] if candidates else ""
if not zip_path:
    try:
        from google.colab import files
        up = files.upload()
        zip_path = "/content/" + next(iter(up))
    except ImportError:
        raise FileNotFoundError("Keine ZIP gefunden: bei Kaggle als Input hinzufügen oder DATASET_ZIP in Zelle 1 setzen.")
print("ZIP:", zip_path)
shutil.rmtree(f"{WORK}/zip", ignore_errors=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(f"{WORK}/zip")
DATA = locate_dataset(f"{WORK}/zip")
count, minutes, errors = dataset_info(DATA)
print(f"{count} Aufnahmen, {minutes:.1f} Minuten")
for e in errors[:20]:
    print("FEHLER", e)
if errors:
    raise SystemExit("Bitte die Fehler beheben (im Stimmen-Studio unter Prüfen) und die ZIP neu erzeugen.")
if minutes < 15:
    raise SystemExit(f"Nur {minutes:.1f} Minuten: zu wenig. Erst weitere Sätze aufnehmen (Ziel 30 bis 60 Minuten).")
if minutes < 30:
    print(f"WARNUNG: {minutes:.1f} Minuten ist knapp; das Ergebnis wird mit mehr Aufnahmen deutlich besser.")

## 6. Ausgangsstimme laden (etwa 900 MB)

In [ ]:
import torch
from huggingface_hub import hf_hub_download, list_repo_files

REPO_ID = "rhasspy/piper-checkpoints"
files = [f for f in list_repo_files(REPO_ID, repo_type="dataset") if f.startswith(BASE + "/") and f.endswith(".ckpt")]
if not files:
    raise FileNotFoundError(f"Kein Checkpoint unter {BASE} gefunden. Ordner prüfen: huggingface.co/datasets/{REPO_ID}")
BASE_CKPT = hf_hub_download(REPO_ID, files[0], repo_type="dataset")
base_epoch = int(torch.load(BASE_CKPT, map_location="cpu", weights_only=False)["epoch"])
MAX_EPOCHS = base_epoch + EXTRA_EPOCHS
print(f"Ausgangsstimme {files[0]}: Epoche {base_epoch} -> trainiere bis Epoche {MAX_EPOCHS} (+{EXTRA_EPOCHS})")

## 7. Trainieren

Die Ausgabe zeigt den Fortschritt. Die Zelle läuft, bis `EXTRA_EPOCHS` erreicht sind oder `MAX_HOURS` abgelaufen ist (dann wird gespeichert).
Du kannst sie auch jederzeit mit **Stopp** unterbrechen und mit Zelle 8 den aktuellen Zwischenstand ausprobieren; später mit `RESUME = True` weitermachen.

In [ ]:
import subprocess

CKPT_ROOT = f"{WORK}/run"
os.makedirs(CKPT_ROOT, exist_ok=True)
start_ckpt = BASE_CKPT
if RESUME:
    latest = newest_checkpoint(CKPT_ROOT)
    if latest:
        start_ckpt = latest
        print("Mache weiter mit", latest)
cmd = [
    sys.executable, "-m", "piper.train", "fit",
    "--data.voice_name", NAME,
    "--data.csv_path", f"{DATA}/metadata.csv",
    "--data.audio_dir", f"{DATA}/wavs/",
    "--model.sample_rate", "22050",
    "--data.espeak_voice", ESPEAK_VOICE,
    "--data.cache_dir", f"{WORK}/cache/",
    "--data.config_path", f"{WORK}/{NAME}.onnx.json",
    "--data.batch_size", str(BATCH),
    "--ckpt_path", start_ckpt,
    "--trainer.max_epochs", str(MAX_EPOCHS),
    "--trainer.max_time", f"00:{int(MAX_HOURS):02d}:00:00",
]
print("$", " ".join(cmd))
proc = subprocess.Popen(cmd, cwd=CKPT_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
try:
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
except KeyboardInterrupt:
    proc.terminate()
    print("Unterbrochen: der letzte Zwischenstand bleibt erhalten.")
print("Rückgabewert:", proc.returncode)
print("Neuester Zwischenstand:", newest_checkpoint(CKPT_ROOT))

## 8. Exportieren, verpacken, anhören

Kann auch mit einem **Zwischenstand** laufen (nach Stopp oder Zeitlimit), um früh zu hören, wie die Stimme klingt.

In [ ]:
import subprocess, urllib.request, tarfile

ckpt = newest_checkpoint(CKPT_ROOT)
assert ckpt, "Noch kein Zwischenstand vorhanden: erst Zelle 7 laufen lassen."
raw_onnx = f"{WORK}/{NAME}.onnx"
sh(f"{sys.executable} -m piper.train.export_onnx --checkpoint '{ckpt}' --output-file '{raw_onnx}'")

espeak_tar = f"{WORK}/espeak-ng-data.tar.bz2"
if not os.path.isdir(f"{WORK}/espeak-ng-data"):
    urllib.request.urlretrieve("https://github.com/k2-fsa/sherpa-onnx/releases/download/tts-models/espeak-ng-data.tar.bz2", espeak_tar)
    with tarfile.open(espeak_tar) as t:
        t.extractall(WORK)
OUT = f"{WORK}/fertig"
shutil.rmtree(OUT, ignore_errors=True)
FOLDER = package_voice(raw_onnx, f"{WORK}/{NAME}.onnx.json", NAME, LANG, OUT, espeak_dir=f"{WORK}/espeak-ng-data", int8=INT8)
print("Stimmenordner:", FOLDER, "-", sorted(os.listdir(FOLDER)))

import sherpa_onnx
onnx_file = glob.glob(f"{FOLDER}/*.onnx")[0]
tts = sherpa_onnx.OfflineTts(sherpa_onnx.OfflineTtsConfig(model=sherpa_onnx.OfflineTtsModelConfig(
    vits=sherpa_onnx.OfflineTtsVitsModelConfig(model=onnx_file, tokens=f"{FOLDER}/tokens.txt", data_dir=f"{FOLDER}/espeak-ng-data"), num_threads=2)))
audio = tts.generate(TEST_SENTENCE, sid=0, speed=1.0)
import numpy as np
pcm = (np.clip(np.array(audio.samples), -1, 1) * 32767).astype("<i2")
test_wav = f"{FOLDER}/test.wav"
with wave.open(test_wav, "wb") as w:
    w.setnchannels(1); w.setsampwidth(2); w.setframerate(audio.sample_rate); w.writeframes(pcm.tobytes())
from IPython.display import Audio, display
display(Audio(test_wav))

## 9. Ergebnis herunterladen

Das ZIP enthält den fertigen Stimmenordner. Auf dem Handy: ZIP entpacken, den Ordner aufs Handy kopieren, in der Chat-App
**Einstellungen → Piper-Stimmen → Stimmenordner wählen…**, danach im Companion-Editor unter **Stimme** auswählen.

In [ ]:
archive = shutil.make_archive(f"{WORK}/{os.path.basename(FOLDER)}", "zip", OUT)
if os.path.isdir("/kaggle/working"):
    shutil.copy(archive, "/kaggle/working/")
    print("Kaggle: die Datei liegt im Reiter Output (rechts) bzw. nach 'Save Version' unter Output:", os.path.basename(archive))
else:
    from google.colab import files
    files.download(archive)
print(archive, round(os.path.getsize(archive) / 1e6), "MB")